# Synechocystis MEP model — figures

Figure driver for the model variants in `synmep.models`. Switch `variant` below to
run the same figures with another model version (ADR 0001).

## Setup

In [ ]:
import copy
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from mxlpy import Simulator, mca, scan
from mxlpy.integrators import Native
from mxlpy.parallel import parallelise
from mxlpy.runners import KineticRunner

from synmep.data import CELL_VOLUME, DAY_IN_MIN, load_legacy_inputs
from synmep.models import v0_faithful as variant
from synmep.plot import savefig

INPUTS = load_legacy_inputs()
NATIVE = partial(Native, method="CVODE_BDF")
RUNNER = KineticRunner(integrator=NATIVE)
EXP_GROWTH_RATE_BY_DENSITY = INPUTS.exp_growth_rate_by_density


def get_model(strain="dshc", growth_mode="steady"):
    return variant.get_model(INPUTS, strain, growth_mode)


def simulate_to_steady_state(model, y0):
    s = Simulator(model, y0=y0, integrator=NATIVE).simulate_to_steady_state()
    return s.get_result().unwrap_or_err()


def final_state(model, res) -> dict[str, float]:
    """Last values of the model's variables (`res.variables` also holds derived ones)."""
    return res.variables.iloc[-1][model.get_variable_names()].to_dict()


def generate_initial_conditions() -> dict[str, float]:
    ms = get_model("dshc", "steady")
    y0 = ms.get_initial_conditions() | {
        "Cell_density": EXP_GROWTH_RATE_BY_DENSITY.index[24] / 1e6
    }
    return final_state(ms, simulate_to_steady_state(ms, y0))


Y0_SS = generate_initial_conditions()

## Fig ??: compare min kcat with literature

In [ ]:
with plt.rc_context({"lines.linewidth": 3, "font.size": 12.0, "axes.grid": True}):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.set_axisbelow(b=True)
    kcat_comp = pd.DataFrame.from_dict(INPUTS.min_kcat_values, "index")
    for key, value in INPUTS.enzyme_parameters.items():
        if key in kcat_comp.index:
            kcat_comp.loc[key, "A"] = value
    kcat_comp.plot(kind="bar", ax=ax)
    ax.legend([r"Calculated min $\mathrm{k_{cat}}$", r"Literature $\mathrm{k_{cat}}$"])

## Fig 09: growth rate over time

Batch growth in the MC-1000-MIX at 80 µE m⁻² s⁻¹, sample depth 13.5 mm (half the tube
diameter), starting from the pre-culture density (paper Fig. 9).

In [ ]:
def make_fig9_data(y0_ss: dict[str, float]) -> pd.DataFrame:
    md = get_model("dshc", "dynamic")
    pars = {"light": 80, "sample_depth_m": 0.0135, "ShcKcat": 0, "Shc_ksynth": 0}
    pars |= {"DXS_inh_str": 0}
    md.update_parameters({k: v for k, v in pars.items() if k in md.get_parameter_names()})
    s = Simulator(
        md, y0=y0_ss | {"Cell_density": 54562873.0217531 / 1e6}, integrator=NATIVE
    ).simulate(96 * 60, steps=96 * 4)
    return s.get_result().unwrap_or_err().get_combined()


res_fig9 = make_fig9_data(Y0_SS)

In [ ]:
def fig9(res: pd.DataFrame) -> None:
    with plt.rc_context({"lines.linewidth": 3, "font.size": 12.0}):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")
        (res["Growth_rate"] * 60).plot(ax=ax1)
        ax1.scatter(
            x=np.linspace(0, 96 * 60, len(EXP_GROWTH_RATE_BY_DENSITY)),
            y=EXP_GROWTH_RATE_BY_DENSITY,
            c="C1",
        )
        ax1.set_ylabel("Growth rate $[h^{-1}]$")
        (res["Cell_density"] * 1e6).plot(ax=ax2, label="Simulated values")
        ax2.plot(
            np.linspace(0, 96 * 60, len(EXP_GROWTH_RATE_BY_DENSITY[::2])),
            EXP_GROWTH_RATE_BY_DENSITY[::2].index,
            label="Experimental values",
        )
        ax2.set_yscale("log")
        ax2.set_ylabel(r"Cell density $[Cells\ mL^{-1}]$")
        ax2.legend(loc="lower right")
        for ax in (ax1, ax2):
            ax.set_xlabel("Time [h]")
            ax.set_xticks(np.arange(0, 97 * 60, 24 * 60))
            ax.set_xticklabels(np.arange(0, 97, 24))
            ax.grid()
        savefig(fig, "fig9-growth.png")
        plt.show()


fig9(res_fig9)

## Fig 10: steady-state growth rate by cell density

80 µE m⁻² s⁻¹, sample depth 2 cm.

In [ ]:
def make_fig10_data() -> pd.Series:
    ms = get_model("dshc", "steady")
    ms.update_parameters({"light": 80, "sample_depth_m": 0.02})
    dens = EXP_GROWTH_RATE_BY_DENSITY.index[::2]
    res = scan.steady_state(
        ms,
        to_scan=pd.DataFrame({"Cell_density": np.asarray(dens) / 1e6}),
        parallel=False,  # one native compile beats one per worker for short scans
        runner=RUNNER,
    )
    return pd.Series(res.get_args()["Growth_rate"].to_numpy() * 60, index=dens)


growth_rate_by_density = make_fig10_data()

In [ ]:
def fig10(growth_rate_by_density: pd.Series) -> None:
    with plt.rc_context({"lines.linewidth": 3, "font.size": 12.0, "axes.grid": True}):
        fig, ax = plt.subplots(figsize=(4, 3))
        growth_rate_by_density.plot(ax=ax, label="Simulation")
        EXP_GROWTH_RATE_BY_DENSITY.plot(ax=ax, marker="o", label="Experimental")
        ax.set_xscale("log")
        ax.set_ylim(0)
        ax.set_ylabel(r"Growth rate [$h^{-1}$]")
        ax.set_xlabel(r"Cell density [$Cells\ ml^{-1}$]")
        ax.legend()
        savefig(fig, "fig10-growth-by-density.png")
        plt.show()


fig10(growth_rate_by_density)

## Fig 11: time point of CrtE repression

CrtE knock-down (`CrtE3Kcat` → 0.01) induced at different time points; state after
14 days.

In [ ]:
def product_after_induction(
    time_point_of_induction: float,
    t_end: float,
    y0_ss: dict[str, float],
) -> pd.Series:
    # fresh model per run: Simulator.update_parameter mutates the model
    s = Simulator(get_model("dshc", "dynamic"), y0=y0_ss, integrator=NATIVE)
    if time_point_of_induction > 0:
        s.simulate(time_point_of_induction)
    s.update_parameter("CrtE3Kcat", 0.01)
    s.simulate(t_end)
    return s.get_result().unwrap_or_err().get_combined().iloc[-1]


def make_fig11_data(y0: dict[str, float]) -> pd.DataFrame:
    t_end = 14 * DAY_IN_MIN
    pts = np.arange(0, t_end, DAY_IN_MIN / 2, dtype=float)
    res = parallelise(
        partial(product_after_induction, y0_ss=y0, t_end=t_end),
        list(zip(pts, pts, strict=True)),
        parallel=False,
    )
    return pd.DataFrame(dict(res)).T.sort_index()


res_fig11 = make_fig11_data(Y0_SS)

In [ ]:
def fig11(res: pd.DataFrame) -> None:
    with plt.rc_context({"lines.linewidth": 3, "font.size": 12.0, "axes.grid": True}):
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(8, 3), layout="constrained")
        fig.suptitle("Total concentration after 14 days")
        days = res.index / DAY_IN_MIN
        # cells nL^-1 -> cells L^-1; mM SQ x 410.73 g/mol x L/cell -> mg L^-1 culture
        ax1.plot(days, res["Cell_density"] * 1e9 * res["SQ"] * 410.73 * CELL_VOLUME)
        ax1.set_ylabel("Squalene conc. in culture\n" r"[$mg\ L^{-1}$]")
        ax2.plot(days, res["Cell_density"] * 1e9)
        ax2.set_yscale("log")
        ax2.set_ylabel(r"Cell density [$Cells\ L^{-1}$]")
        ax3.plot(days, res["SQ"] * 410.73 / 1000)
        ax3.set_ylabel("Intracellular squalene\n" r"concentration [$g\ L^{-1}$]")
        for ax in (ax1, ax2, ax3):
            ax.set_xlabel("Timepoint of induction [d]")
        savefig(fig, "fig11-induction.png")
        plt.show()


fig11(res_fig11)

## Fig 12: iterative MCA, no repeated overexpression

Each round: flux response coefficients of all enzyme synthesis rates (`*_ksynth`);
overexpress the enzyme with the largest summed response of squalene synthesis and
biomass, by the rhamnose-promoter overexpression potential seen in the sqs-strain
proteomics.

In [ ]:
def make_fig12_data(
    poi: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    ms = get_model("dshc", "steady")
    y0 = ms.get_initial_conditions()
    pars_for_mca = [p for p in ms.get_parameter_names() if "ksynth" in p]
    frcs_by_run, fluxes_by_run, overexpressed = {}, {}, []

    for run in range(10):
        rc = mca.response_coefficients(
            ms, to_scan=pars_for_mca, variables=y0, parallel=False, runner=RUNNER
        )
        frc = rc.fluxes.T  # rows: parameters, columns: fluxes (modelbase layout)
        frcs_by_run[run] = frc
        res = simulate_to_steady_state(ms, y0)
        fluxes_by_run[run] = res.fluxes.iloc[-1]
        y0 = final_state(ms, res)

        frc_filtered = frc.drop(index=overexpressed)
        most_positive = (frc_filtered["v_Sqs"] + frc_filtered["v_Biomass"]).idxmax()
        best = poi[poi["Enzyme"] == most_positive.removesuffix("_ksynth")]["kDa"]
        if len(best) == 0:
            break
        overexpression_amount = 1.331057 * best.iloc[0]
        ms.update_parameter(
            most_positive,
            ms.get_parameter_values()[most_positive] + overexpression_amount / (5.5 * 60),
        )
        overexpressed.append(most_positive)
    return (
        pd.concat(frcs_by_run),
        pd.concat(fluxes_by_run).unstack(),
        overexpressed,
    )


frc_by_run, fluxes_by_run, overexpressed = make_fig12_data(INPUTS.proteins_of_interest)

In [ ]:
def fig12(frc_by_run, fluxes_by_run, overexpressed) -> None:
    with plt.rc_context({"lines.linewidth": 3}):
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(7, 5), layout="constrained", height_ratios=(5, 1), sharex=True
        )
        data = (
            frc_by_run["v_Sqs"].unstack().T
            + frc_by_run["v_Biomass"].unstack().min(axis=1).abs()
        )
        sns.heatmap(data, annot=False, cmap="RdBu_r", center=0, ax=ax1)
        ax1.set_ylabel("Adjustable parameters", fontsize=12)
        for run, gene in enumerate(overexpressed):
            ax1.add_patch(
                plt.Rectangle(
                    (run, data.index.get_loc(gene)), 1, 1,
                    fill=False, edgecolor="red", linewidth=3,
                )
            )
        ax2.plot(fluxes_by_run["v_Sqs"] / fluxes_by_run["v_Biomass"] * 410.73 / 1000)
        ax2.set_ylim(0)
        ax2.grid(True)
        ax2.set_xlabel("No. of genes overexpressed")
        ax2.set_ylabel("Intracellular squalene \n" r"concentration [$g\ L^{-1}$]", rotation=0)
        ax2.yaxis.set_label_coords(-0.35, 0.15)
        savefig(fig, "fig12-overexpression.png")
        plt.show()


fig12(frc_by_run, fluxes_by_run, overexpressed)